In [1]:
!pip install azure-eventhub

StatementMeta(, bbb1cfa6-93e1-4f87-bf93-6ed2796a82a0, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 12.0 MB/s eta 0:00:00


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, expr
from pyspark.sql.types import StructType, StructField, StringType

# Initialize Spark Session
spark = SparkSession.builder.appName("Zava_MarketingSignals").getOrCreate()

# Define the schema
schema = StructType([
    StructField("Competitor", StringType(), True),
    StructField("CustomerTier", StringType(), True),
    StructField("CampaignName", StringType(), True),
    StructField("InterestRateChange", StringType(), True),
    StructField("PreApprovedLoanAmount", StringType(), True)
])

# Full data mapping (6 Banks x 4 Tiers)
# data = [
#     # Platinum Tier
#     ("XYZ Bank", "Platinum", "Elite Mortgage Elite", "-0.25% (Now 5.75%)", "$250,000"),
#     ("ABC Bank", "Platinum", "Private Client Premier", "-0.30% (Now 5.65%)", "$300,000 - $600,000"),
#     ("XYZ Lending", "Platinum", "Executive Refi-Plus", "-0.15% (Now 5.80%)", "$200,000 - $450,000"),
#     ("ABC Lending", "Platinum", "Platinum Peak Equity", "-0.20% (Now 5.70%)", "$250,000 - $550,000"),
#     ("XYZ Finance", "Platinum", "Wealth Anchor Elite", "-0.10% (Now 5.85%)", "$220,000 - $480,000"),
#     ("ABC Finance", "Platinum", "Premier Legacy Loan", "-0.05% (Now 5.90%)", "$200,000 - $400,000"),

#     # Gold Tier
#     ("XYZ Bank", "Gold", "Gold Advantage", "-0.15% (Now 6.25%)", "$150,000 - $200,000"),
#     ("ABC Bank", "Gold", "Preferred Gold Home", "-0.10% (Now 6.15%)", "$175,000 - $275,000"),
#     ("XYZ Lending", "Gold", "Wealth-Builder Special", "-0.20% (Now 6.10%)", "$125,000 - $225,000"),
#     ("ABC Lending", "Gold", "Gold Standard Equity", "-0.05% (Now 6.40%)", "$140,000 - $240,000"),
#     ("XYZ Finance", "Gold", "Strategic Growth Gold", "+0.05% (Now 6.50%)", "$100,000 - $200,000"),
#     ("ABC Finance", "Gold", "Equity Direct Gold", "No Change (6.40%)", "$100,000 - $200,000"),

#     # Silver Tier
#     ("XYZ Bank", "Silver", "Mid-Market Growth", "+0.10% (Now 7.15%)", "$50,000 - $125,000"),
#     ("ABC Bank", "Silver", "Silver Loyalty Path", "-0.05% (Now 7.05%)", "$75,000 - $150,000"),
#     ("XYZ Lending", "Silver", "Streamline Silver", "No Change (7.45%)", "$40,000 - $100,000"),
#     ("ABC Lending", "Silver", "Silver Direct Access", "+0.15% (Now 7.60%)", "$60,000 - $110,000"),
#     ("XYZ Finance", "Silver", "Silver Flex Finance", "-0.10% (Now 7.25%)", "$55,000 - $130,000"),
#     ("ABC Finance", "Silver", "Standard Silver Home", "No Change (7.30%)", "$50,000 - $100,000"),

#     # Bronze Tier
#     ("XYZ Bank", "Bronze", "Entry-Level Equity", "+0.25% (Now 8.75%)", "$10,000 - $40,000"),
#     ("ABC Bank", "Bronze", "Step-Up Bronze", "+0.10% (Now 8.50%)", "$15,000 - $45,000"),
#     ("XYZ Lending", "Bronze", "Fresh Start Finance", "+0.15% (Now 8.95%)", "$15,000 - $35,000"),
#     ("ABC Lending", "Bronze", "Starter Home Special", "No Change (9.10%)", "$10,000 - $30,000"),
#     ("XYZ Finance", "Bronze", "Basic Builder Credit", "+0.20% (Now 9.25%)", "$5,000 - $25,000"),
#     ("ABC Finance", "Bronze", "Simple Step Credit", "No Change (9.20%)", "$5,000 - $25,000")
# ]
data = [
    # Platinum
    ("XYZ Bank", "Platinum", "Elite Mortgage Elite", 5.75, 800000),
    ("ABC Bank", "Platinum", "Private Client Premier", 5.65, 750000),
    ("XYZ Lending", "Platinum", "Executive Refi Plus", 5.80, 720000),

    # Gold
    ("XYZ Bank", "Gold", "Gold Advantage", 6.25, 180000),
    ("ABC Bank", "Gold", "Preferred Gold Home", 6.15, 200000),

    # Silver
    ("XYZ Bank", "Silver", "Mid Market Growth", 7.15, 100000),
    ("ABC Bank", "Silver", "Silver Loyalty Path", 7.05, 120000),

    # Bronze
    ("XYZ Bank", "Bronze", "Entry Level Equity", 8.75, 40000),
    ("ABC Bank", "Bronze", "Step Up Bronze", 8.50, 45000)
]
# Create DataFrame
raw_df = spark.createDataFrame(data, schema)

# Add SignalID (UUID) and SignalTimestamp (current time)
marketing_signals_df = raw_df \
    .withColumn("SignalID", expr("uuid()")) \
    .withColumn("SignalTimestamp", current_timestamp()) \
    .select("SignalID", "SignalTimestamp", "Competitor", "CustomerTier", "CampaignName", "InterestRateChange", "PreApprovedLoanAmount")

# Show the results
display(marketing_signals_df)


StatementMeta(, bbb1cfa6-93e1-4f87-bf93-6ed2796a82a0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d61781e-072c-4072-97a5-650a90f94fe4)

In [ ]:
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime, date
from decimal import Decimal
import json

# Serializer
def json_serializer(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Decimal):
        return float(obj)
    raise TypeError(f"Type {type(obj)} not serializable")


CONNECTION_STR = "###CONNECTION_STR###"
EVENT_HUB_NAME = "###EVENT_HUB_NAME###"

rows = marketing_signals_df.collect()
print(f"Preparing to send {len(rows)} events...")

producer = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR, eventhub_name=EVENT_HUB_NAME)

sent_count = 0
try:
    event_data_batch = producer.create_batch()
    for row in rows:
        event = EventData(json.dumps(row.asDict(), default=json_serializer))
        try:
            event_data_batch.add(event)
        except ValueError:
            producer.send_batch(event_data_batch)
            sent_count += len(event_data_batch)
            event_data_batch = producer.create_batch()
            event_data_batch.add(event)

    if len(event_data_batch) > 0:
        producer.send_batch(event_data_batch)
        sent_count += len(event_data_batch)

    print(f"✅ Successfully sent {sent_count} events to Event Hub")
finally:
    producer.close()

StatementMeta(, bbb1cfa6-93e1-4f87-bf93-6ed2796a82a0, 5, Finished, Available, Finished, False)

Preparing to send 9 events...
✅ Successfully sent 9 events to Event Hub


In [ ]:
workspace_name = "###WORKSPACE_NAME###"
lakehouse_name = "###LAKEHOUSE_NAME###"
table_path = f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{lakehouse_name}.Lakehouse/Tables/dbo/transactions"
df_stream = spark.read.format("delta").load(table_path)

In [ ]:
from pyspark.sql.types import DecimalType, DoubleType
from pyspark.sql import functions as F
from datetime import datetime, date
from decimal import Decimal

from pyspark.sql.functions import col

df_stream = df_stream.select(
    col("TransID").cast("string"),
    col("CustomerID").cast("string"),
    col("Amount").cast("double"),
    col("Category").cast("string"),
    col("Description").cast("string"),
    col("Timestamp").cast("timestamp")
)


# Convert Decimal columns to Double
for field in df_stream.schema.fields:
    if isinstance(field.dataType, DecimalType):
        df_stream = df_stream.withColumn(
            field.name,
            F.col(field.name).cast(DoubleType())
        )

# Eventhouse details
kustoUri = "###KUSTO_URI###"
database = "###KQL_DB_NAME###"
table_name = "realtime_transactions"

# Get Fabric token
accessToken = mssparkutils.credentials.getToken(kustoUri)

# Write directly to Eventhouse
df_stream.write \
    .format("com.microsoft.kusto.spark.synapse.datasource") \
    .option("accessToken", accessToken) \
    .option("kustoCluster", kustoUri) \
    .option("kustoDatabase", database) \
    .option("kustoTable", table_name) \
    .option("tableCreateOptions", "CreateIfNotExist") \
    .mode("append") \
    .save()

print(f"Loaded data into {database}.{table_name}")